# Timor-Leste: WorldPop 2026 full exact Pareto replication

This notebook reconstructs the component-aware Timor-Leste maximum-cover study from open geospatial inputs and computes **nine complete exact Pareto frontiers with Gurobipy**:

- candidate grids: 10 km, 5 km, and 1 km;
- accessibility thresholds: 2 km, 5 km, and 10 km;
- every integer budget from zero through the first exact saturation budget;
- one complete selected-candidate solution and detailed timing record for every budget.

The geographic pipeline constructs sparse facility-to-demand coverage. The optimization layer consumes only that sparse relation and population weights. All exact optimization in this notebook uses Gurobipy directly.

## Reproduction contract

The published numerical results use WorldPop Global2 `R2025A v1`, constrained 100 m, reference date `2026-01-01`, and the fixed Timor-Leste Geofabrik PBF identified below. OpenStreetMap evolves continuously; replacing the PBF is a new data vintage rather than a bit-for-bit reproduction.

| input | exact identifier |
|---|---|
| WorldPop | `tls_pop_2026_CN_100m_R2025A_v1.tif` |
| WorldPop SHA-256 | `00ba7043d74191fb912c9fa3d6c6470cca17f156433a9eef61c85b5ccfa62436` |
| OSM PBF SHA-256 | `bf5388ca42a86650ce0d4b3513b18752728b10a96cbaa5e8112a3729ed1247d1` |
| pipeline commit used for matrices | `1be52321f1f218f0a11f7faf420ae007fcd67d72` |
| first `abw_maxcover` commit with durable Gurobipy callbacks | `499b8dc3ba01f8bd712990eb3feab850e37c4b9d` |

The pipeline retains WorldPop cells with population at least one, uses the unsimplified OSM driving network, OSM health amenities, component-aware snapping to components 0 and 1, and computes road distances up to 10 km.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPOSITORY_URL = "https://github.com/Analytics-for-a-Better-World/Public-Infrastructure-Service-Access.git"
PIPELINE_COMMIT = "1be52321f1f218f0a11f7faf420ae007fcd67d72"
ABW_CALLBACK_COMMIT = "499b8dc3ba01f8bd712990eb3feab850e37c4b9d"

WORK_ROOT = Path(os.environ.get(
    "PARVATHY_REPLICATION_ROOT",
    r"C:\work\codex\sandboxes\Conclude_Parvathy_thesis",
))
REPOSITORY = Path(os.environ.get(
    "PISA_REPOSITORY",
    r"C:\github\Public-Infrastructure-Service-Access",
))
PIPELINE_RUN_ROOT = WORK_ROOT / "runs" / "timor_global2_2026_20260717_clean"
DATA_DIR = PIPELINE_RUN_ROOT / "east-timor_data"
PIPELINE_OUTPUT = DATA_DIR / "outputs"
PARETO_OUTPUT = WORK_ROOT / "outputs" / "timor_global2_2026_exact_pareto_20260717"
PROJECT_DIR = REPOSITORY / "Research-Sandbox" / "Parvathy_PhD" / "Timor_Leste"
CAMPAIGN_SCRIPT = PROJECT_DIR / "tools" / "run_timor_global2_2026_exact_pareto.py"

# Explicit switches prevent an accidental multi-hour rerun when the notebook is opened.
# Set them to True for a clean end-to-end reproduction. Existing valid caches are reused.
RUN_PIPELINE = False
RUN_GUROBI = False
ALLOW_INPUT_DRIFT = False

WORLDPOP_URL = (
    "https://worldpop-public-data.soton.ac.uk/GIS/Population/"
    "Global_2015_2030/R2025A/2026/TLS/v1/100m/constrained/"
    "tls_pop_2026_CN_100m_R2025A_v1.tif"
)
WORLDPOP_SHA256 = "00ba7043d74191fb912c9fa3d6c6470cca17f156433a9eef61c85b5ccfa62436"
PBF_URL = "https://download.geofabrik.de/asia/east-timor-latest.osm.pbf"
PBF_SHA256 = "bf5388ca42a86650ce0d4b3513b18752728b10a96cbaa5e8112a3729ed1247d1"
PBF_SNAPSHOT_SOURCE = Path(os.environ.get(
    "TIMOR_PBF_SNAPSHOT",
    str(DATA_DIR / "east-timor-latest.osm.pbf"),
))

PARETO_OUTPUT.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version)
print("Repository:", REPOSITORY)
print("Pipeline output:", PIPELINE_OUTPUT)
print("Pareto output:", PARETO_OUTPUT)

## Obtain and verify the code

On a new machine, clone the repository before running the remaining cells. The notebook checks that the checkout contains the committed Gurobipy callback used for durable per-budget checkpoints. The matrix manifest itself records the older pipeline commit used for the published data instance.

In [ ]:
if not (REPOSITORY / ".git").exists():
    REPOSITORY.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY)], check=True)

head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPOSITORY, text=True).strip()
contains_callback = subprocess.run(
    ["git", "merge-base", "--is-ancestor", ABW_CALLBACK_COMMIT, head],
    cwd=REPOSITORY,
    check=False,
).returncode == 0
if not contains_callback:
    raise RuntimeError(
        f"Repository {head} does not contain required abw_maxcover commit {ABW_CALLBACK_COMMIT}. "
        "Fetch or update main before continuing."
    )
print("Repository HEAD:", head)
print("Required Gurobipy checkpoint support present:", contains_callback)

## Acquire and verify the inputs

The WorldPop raster is immutable at its versioned URL. Geofabrik's `latest` URL is not immutable. For bit-for-bit reproduction, point `TIMOR_PBF_SNAPSHOT` to the archived PBF with the recorded SHA-256. If only the current PBF is available, set `ALLOW_INPUT_DRIFT=True`; the resulting study must be labeled with its new PBF hash and acquisition date.

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download(url: str, destination: Path) -> None:
    import requests
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".partial")
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with temporary.open("wb") as handle:
            for block in response.iter_content(1024 * 1024):
                if block:
                    handle.write(block)
    temporary.replace(destination)

DATA_DIR.mkdir(parents=True, exist_ok=True)
worldpop_path = DATA_DIR / "tls_pop_2026_CN_100m_R2025A_v1.tif"
pbf_path = DATA_DIR / "east-timor-latest.osm.pbf"

if not worldpop_path.exists():
    download(WORLDPOP_URL, worldpop_path)
if not pbf_path.exists():
    if PBF_SNAPSHOT_SOURCE.exists():
        shutil.copy2(PBF_SNAPSHOT_SOURCE, pbf_path)
    else:
        download(PBF_URL, pbf_path)

observed_worldpop_hash = sha256(worldpop_path)
observed_pbf_hash = sha256(pbf_path)
print("WorldPop SHA-256:", observed_worldpop_hash)
print("PBF SHA-256:", observed_pbf_hash)
if observed_worldpop_hash != WORLDPOP_SHA256:
    raise RuntimeError("WorldPop raster hash mismatch")
if observed_pbf_hash != PBF_SHA256 and not ALLOW_INPUT_DRIFT:
    raise RuntimeError(
        "OSM PBF hash differs from the published snapshot. Supply TIMOR_PBF_SNAPSHOT "
        "or explicitly allow a new data vintage."
    )

## Build the sparse geographic instances

This is the complete public pipeline invocation. Each run writes a manifest containing the input URLs/hashes, parameters, output paths, and code commit. Runs are sequential so their wall times remain interpretable.

In [ ]:
pipeline_script = (
    REPOSITORY
    / "Research-Sandbox"
    / "general_distances_per_country"
    / "run_pipeline.py"
)

grid_cases = [(10_000, 5_000), (5_000, 2_500), (1_000, 500)]
if RUN_PIPELINE:
    for spacing_m, maximum_snap_m in grid_cases:
        command = [
            sys.executable,
            str(pipeline_script),
            "timor-leste",
            "--data-root", str(PIPELINE_RUN_ROOT),
            "--cache-root", str(DATA_DIR / "cache"),
            "--output-root", str(PIPELINE_OUTPUT),
            "--pbf-filename", pbf_path.name,
            "--network-backend", "osmium",
            "--network-profile", "driving",
            "--simplify-network", "false",
            "--diagnose-connectivity", "true",
            "--snap-components", "0,1",
            "--sources", "amenities", "candidates",
            "--destinations", "population",
            "--candidate-grid-spacing-m", str(spacing_m),
            "--candidate-max-snap-dist-m", str(maximum_snap_m),
            "--candidate-exclude-water", "false",
            "--max-total-dist", "10000",
            "--matrix-output-mode", "split",
            "--matrix-shape", "sparse",
            "--population-provider", "worldpop",
            "--worldpop-dataset", "global2",
            "--worldpop-year", "2026",
            "--worldpop-release", "R2025A",
            "--worldpop-version", "v1",
            "--worldpop-resolution", "100m",
            "--worldpop-constrained", "true",
            "--worldpop-url", WORLDPOP_URL,
        ]
        print("Running", spacing_m, "m candidate grid")
        subprocess.run(command, check=True)
else:
    print("RUN_PIPELINE=False: using existing manifests and sparse matrices.")

manifests = sorted(PIPELINE_OUTPUT.glob("run_manifest*.yaml"))
if len(manifests) < 3:
    raise RuntimeError("Expected candidate-grid pipeline manifests are missing")
print("Pipeline manifests available:", len(manifests))

## Compute all exact Pareto frontiers with Gurobipy

`abw_maxcover` constructs a reusable Gurobi model for each of the nine instances. Budgets are solved in ascending order. The previous optimum plus incremental greedy additions provides the next MIP start. Every returned budget is immediately appended to `frontier.jsonl` and `solutions.jsonl` with an `fsync`, so rerunning this cell resumes after the last verified optimal budget.

The default limit is one hour **per budget**, with relative MIP gap `1e-9`. Every retained row must have status `optimal`; a time limit stops that case rather than silently treating an incumbent as exact.

In [ ]:
if not CAMPAIGN_SCRIPT.exists():
    raise FileNotFoundError(CAMPAIGN_SCRIPT)

if RUN_GUROBI:
    environment = os.environ.copy()
    environment.update({
        "PARVATHY_REPLICATION_ROOT": str(WORK_ROOT),
        "PISA_REPOSITORY": str(REPOSITORY),
        "TIMOR_GLOBAL2_2026_PIPELINE_OUTPUT": str(PIPELINE_OUTPUT),
        "TIMOR_GLOBAL2_2026_PARETO_OUTPUT": str(PARETO_OUTPUT),
    })
    subprocess.run(
        [
            sys.executable,
            str(CAMPAIGN_SCRIPT),
            "--time-limit", "3600",
            "--mip-gap", "1e-9",
        ],
        check=True,
        env=environment,
    )
else:
    print("RUN_GUROBI=False: validating the existing exact checkpoints.")

## Read and validate every frontier and solution

The checks below require contiguous budgets beginning at zero, optimal Gurobi status, monotone objectives, a solution for every frontier point, valid candidate IDs, and equality between the last exact objective and the all-candidate ceiling.

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def latest_by_budget(path: Path, *, require_optimal: bool = True) -> dict[int, dict]:
    latest = {}
    for record in read_jsonl(path):
        if not require_optimal or record.get("status") == "optimal":
            latest[int(record["budget"])] = record
    return latest

case_root = PARETO_OUTPUT / "cases"
summary_rows = []
validation_rows = []
for case_dir in sorted(path for path in case_root.iterdir() if path.is_dir()):
    metadata = json.loads((case_dir / "instance_metadata.json").read_text(encoding="utf-8"))
    frontier = latest_by_budget(case_dir / "frontier.jsonl")
    solutions = latest_by_budget(case_dir / "solutions.jsonl")
    budgets = sorted(frontier)
    objectives = [int(frontier[budget]["incremental_objective_scaled"]) for budget in budgets]
    expected_budgets = list(range(budgets[-1] + 1)) if budgets else []
    catalog_ids = set(pd.read_csv(case_dir / "candidate_catalog.csv")["candidate_id"].astype(str))
    solution_ids_valid = all(
        set(map(str, solutions[budget]["candidate_ids"])).issubset(catalog_ids)
        for budget in budgets
    )
    solution_sizes_valid = all(
        len(solutions[budget]["candidate_ids"]) <= budget for budget in budgets
    )
    ceiling = int(metadata["all_candidate_incremental_population_scaled"])
    checks = {
        "case_id": case_dir.name,
        "contiguous_budgets": budgets == expected_budgets,
        "all_status_optimal": all(frontier[b]["status"] == "optimal" for b in budgets),
        "monotone_objective": objectives == sorted(objectives),
        "solution_for_every_budget": set(solutions) == set(frontier),
        "solution_ids_valid": solution_ids_valid,
        "solution_sizes_valid": solution_sizes_valid,
        "saturated": bool(objectives) and objectives[-1] == ceiling,
    }
    validation_rows.append(checks)

    events = read_jsonl(case_dir / "events.jsonl")
    model_seconds = sum(
        float(event["model_seconds"])
        for event in events
        if event.get("event") == "gurobi_model_built"
    )
    exact_wall_seconds = sum(
        float(event["wall_seconds"])
        for event in events
        if event.get("event") == "exact_session_ended"
    )
    solve_seconds = sum(float(frontier[b]["solve_seconds"]) for b in budgets)
    summary_rows.append({
        "case_id": case_dir.name,
        "grid_km": metadata["spacing_m"] / 1000,
        "threshold_km": metadata["service_threshold_m"] / 1000,
        "active_candidates": metadata["candidate_count_active"],
        "arcs": metadata["n_arcs"],
        "baseline_coverage_pct": metadata["baseline_coverage_pct"],
        "saturation_coverage_pct": metadata["all_candidate_coverage_pct"],
        "exact_saturation_budget": budgets[-1] if budgets else None,
        "frontier_points": len(budgets),
        "instance_build_seconds": metadata["instance_build_seconds"],
        "greedy_upper_seconds": metadata["greedy_saturation_seconds"],
        "gurobi_model_seconds": model_seconds,
        "gurobi_solve_seconds": solve_seconds,
        "exact_wall_seconds": exact_wall_seconds,
    })

validation = pd.DataFrame(validation_rows)
summary = pd.DataFrame(summary_rows).sort_values(["threshold_km", "grid_km"], ascending=[True, False])
display(validation)
display(summary)
if not validation.drop(columns="case_id").to_numpy(dtype=bool).all():
    raise AssertionError("At least one exact-frontier validation failed")

summary.to_csv(PARETO_OUTPUT / "exact_frontier_summary.csv", index=False)
validation.to_csv(PARETO_OUTPUT / "exact_frontier_validation.csv", index=False)

## Plot the complete nine-frontier comparison

Solid lines end at exact saturation. Each curve then continues horizontally as a same-color dashed segment to the end of its axis, with a marker at the exact saturation budget.

In [ ]:
colors = {10: "#0072B2", 5: "#D55E00", 1: "#009E73"}
fig, axes = plt.subplots(3, 3, figsize=(12.2, 9.0), constrained_layout=True)

for row, threshold_km in enumerate((2, 5, 10)):
    for col, grid_km in enumerate((10, 5, 1)):
        ax = axes[row, col]
        case_id = f"timor_global2_2026_grid_{grid_km}km_threshold_{threshold_km}km"
        records = latest_by_budget(case_root / case_id / "frontier.jsonl")
        frame = pd.DataFrame(records.values()).sort_values("budget")
        saturation_budget = int(frame["budget"].max())
        saturation_coverage = float(frame.iloc[-1]["coverage_pct"])
        x_end = max(saturation_budget + 1, int(np.ceil(1.06 * saturation_budget)))
        color = colors[grid_km]
        ax.plot(frame["budget"], frame["coverage_pct"], color=color, linewidth=1.7)
        ax.scatter([saturation_budget], [saturation_coverage], color=color, s=24, zorder=3)
        ax.plot(
            [saturation_budget, x_end],
            [saturation_coverage, saturation_coverage],
            color=color,
            linewidth=1.4,
            linestyle="--",
        )
        ax.set_xlim(0, x_end)
        ax.set_title(f"{grid_km} km candidate grid")
        ax.set_xlabel("New facilities")
        if col == 0:
            ax.set_ylabel(f"Coverage (%)\n{threshold_km} km threshold")
        ax.grid(color="#dddddd", linewidth=0.6)
        ax.spines[["top", "right"]].set_visible(False)

figure_dir = PARETO_OUTPUT / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / "timor_global2_2026_exact_pareto_3x3.pdf", bbox_inches="tight")
fig.savefig(figure_dir / "timor_global2_2026_exact_pareto_3x3.png", dpi=220, bbox_inches="tight")
plt.show()

## Recover any exact solution

Every solution is stored using both the compact internal integer indices and stable candidate IDs. The helper below joins a requested budget to its candidate catalog and returns coordinates suitable for mapping or deployment reporting.

In [ ]:
def load_exact_solution(grid_km: int, threshold_km: int, budget: int) -> pd.DataFrame:
    case_id = f"timor_global2_2026_grid_{grid_km}km_threshold_{threshold_km}km"
    case_dir = case_root / case_id
    solutions = latest_by_budget(case_dir / "solutions.jsonl", require_optimal=False)
    if budget not in solutions or solutions[budget].get("status") != "optimal":
        raise KeyError(f"No optimal solution stored for {case_id}, budget {budget}")
    candidate_ids = list(map(str, solutions[budget]["candidate_ids"]))
    catalog = pd.read_csv(case_dir / "candidate_catalog.csv")
    catalog["candidate_id"] = catalog["candidate_id"].astype(str)
    selected = catalog[catalog["candidate_id"].isin(candidate_ids)].copy()
    order = {candidate_id: index for index, candidate_id in enumerate(candidate_ids)}
    selected["solution_order"] = selected["candidate_id"].map(order)
    return selected.sort_values("solution_order")

# Example: the exact p=20 solution for the 1 km grid and 5 km service threshold.
example_solution = load_exact_solution(1, 5, 20)
display(example_solution.head())
print("Selected facilities:", len(example_solution))

## Generate the complete figure set

The figure generator reads only the validated exact checkpoints and the pipeline source catalog. It creates the nine-frontier panel, a direct 5 km-threshold comparison, resolution and runtime summaries, three candidate-grid maps, and a representative exact `p=100` solution on the 1 km grid. Maps use an OpenStreetMap basemap; the figure manifest records any tile error. The complete plotting implementation is embedded in the next code cell so the notebook is independently inspectable; the matching repository module remains available for command-line reuse.

In [ ]:
os.environ.update({
    "PARVATHY_REPLICATION_ROOT": str(WORK_ROOT),
    "TIMOR_GLOBAL2_2026_PIPELINE_OUTPUT": str(PIPELINE_OUTPUT),
    "TIMOR_GLOBAL2_2026_PARETO_OUTPUT": str(PARETO_OUTPUT),
})
print("Figure paths configured for the embedded implementation.")

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter


ROOT = Path(
    os.environ.get(
        "PARVATHY_REPLICATION_ROOT",
        r"C:\work\codex\sandboxes\Conclude_Parvathy_thesis",
    )
)
OUTPUT = Path(
    os.environ.get(
        "TIMOR_GLOBAL2_2026_PARETO_OUTPUT",
        str(ROOT / "outputs" / "timor_global2_2026_exact_pareto_20260717"),
    )
)
CASE_ROOT = OUTPUT / "cases"
FIGURES = OUTPUT / "figures"
PIPELINE_OUTPUT = Path(
    os.environ.get(
        "TIMOR_GLOBAL2_2026_PIPELINE_OUTPUT",
        str(
            ROOT
            / "runs"
            / "timor_global2_2026_20260717_clean"
            / "east-timor_data"
            / "outputs"
        ),
    )
)

GRID_COLORS = {10: "#0072B2", 5: "#D55E00", 1: "#009E73"}
THRESHOLD_COLORS = {2: "#7B2CBF", 5: "#D55E00", 10: "#0072B2"}
CANDIDATE_COLOR = "#0057B8"
EXISTING_COLOR = "#E8590C"
SELECTED_COLOR = "#169B62"


def case_id(grid_km: int, threshold_km: int) -> str:
    return f"timor_global2_2026_grid_{grid_km}km_threshold_{threshold_km}km"


def case_dir(grid_km: int, threshold_km: int) -> Path:
    return CASE_ROOT / case_id(grid_km, threshold_km)


def resolve_sources_path() -> Path:
    override = os.environ.get("TIMOR_GLOBAL2_2026_SOURCES_PATH")
    if override:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    metadata = json.loads(
        (case_dir(1, 5) / "instance_metadata.json").read_text(encoding="utf-8")
    )
    manifest_path = Path(metadata["manifest_path"])
    if manifest_path.exists():
        import yaml

        manifest = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))
        path = Path(manifest["outputs"]["sources"]["path"])
        if path.exists():
            return path

    matches = sorted(PIPELINE_OUTPUT.glob("sources*.parquet"))
    if not matches:
        raise FileNotFoundError(
            "No source catalog found. Set TIMOR_GLOBAL2_2026_SOURCES_PATH."
        )
    return max(matches, key=lambda path: path.stat().st_mtime_ns)


def read_latest_jsonl(path: Path) -> dict[int, dict]:
    records: dict[int, dict] = {}
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            records[int(record["budget"])] = record
    return records


def save(fig: plt.Figure, name: str, *, dpi: int = 240) -> None:
    FIGURES.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES / f"{name}.pdf", bbox_inches="tight", pad_inches=0.05)
    fig.savefig(
        FIGURES / f"{name}.png",
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.05,
        facecolor="white",
    )
    plt.close(fig)


def style_plot(ax: plt.Axes) -> None:
    ax.grid(color="#D9DEE3", linewidth=0.65)
    ax.spines[["top", "right"]].set_visible(False)


def padded_bounds(
    layers: list[gpd.GeoDataFrame], x_fraction: float = 0.035, y_fraction: float = 0.055
) -> tuple[float, float, float, float]:
    bounds = np.vstack([layer.total_bounds for layer in layers])
    xmin, ymin = bounds[:, :2].min(axis=0)
    xmax, ymax = bounds[:, 2:].max(axis=0)
    xpad = max((xmax - xmin) * x_fraction, 1.0)
    ypad = max((ymax - ymin) * y_fraction, 1.0)
    return xmin - xpad, ymin - ypad, xmax + xpad, ymax + ypad


def add_osm_basemap(
    ax: plt.Axes,
    bounds: tuple[float, float, float, float],
    *,
    zoom: int = 8,
    washout: float = 0.18,
) -> str | None:
    import contextily as ctx

    xmin, ymin, xmax, ymax = bounds
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal", adjustable="box")
    ax.set_anchor("C")
    ax.set_facecolor("#E8EDF0")
    error = None
    try:
        ctx.add_basemap(
            ax,
            source=ctx.providers.OpenStreetMap.Mapnik,
            crs="EPSG:3857",
            zoom=zoom,
            attribution_size=4,
        )
    except Exception as exc:
        error = repr(exc)
    if washout:
        ax.axhspan(ymin, ymax, facecolor="white", alpha=washout, zorder=2)
    ax.set_axis_off()
    return error


def load_geography() -> tuple[dict[int, gpd.GeoDataFrame], gpd.GeoDataFrame]:
    candidates = {}
    for grid_km in (10, 5, 1):
        catalog = pd.read_csv(case_dir(grid_km, 10) / "candidate_catalog.csv")
        candidates[grid_km] = gpd.GeoDataFrame(
            catalog,
            geometry=gpd.points_from_xy(catalog["Longitude"], catalog["Latitude"]),
            crs="EPSG:4326",
        ).to_crs(epsg=3857)

    sources = pd.read_parquet(resolve_sources_path())
    amenities = sources.loc[sources["source_type"].astype(str) == "amenities"].copy()
    existing = gpd.GeoDataFrame(
        amenities,
        geometry=gpd.points_from_xy(amenities["Longitude"], amenities["Latitude"]),
        crs="EPSG:4326",
    ).to_crs(epsg=3857)
    return candidates, existing


def draw_candidate_layer(
    ax: plt.Axes,
    candidates: gpd.GeoDataFrame,
    existing: gpd.GeoDataFrame,
) -> None:
    ax.plot(
        candidates.geometry.x,
        candidates.geometry.y,
        linestyle="None",
        marker=".",
        markersize=1.35,
        markeredgewidth=0,
        color=CANDIDATE_COLOR,
        alpha=0.76,
        rasterized=True,
        zorder=3,
    )
    ax.scatter(
        existing.geometry.x,
        existing.geometry.y,
        marker="D",
        s=15,
        facecolor=EXISTING_COLOR,
        edgecolor="white",
        linewidth=0.45,
        alpha=0.94,
        zorder=5,
    )


def make_candidate_maps() -> list[str]:
    candidates, existing = load_geography()
    bounds = padded_bounds(list(candidates.values()) + [existing])
    errors: list[str] = []

    fig, axes = plt.subplots(3, 1, figsize=(7.8, 7.8), gridspec_kw={"hspace": 0.035})
    for ax, grid_km in zip(axes, (10, 5, 1)):
        error = add_osm_basemap(ax, bounds)
        if error:
            errors.append(error)
        draw_candidate_layer(ax, candidates[grid_km], existing)
        ax.text(
            0.015,
            0.97,
            f"{grid_km} km grid: {len(candidates[grid_km]):,} candidates",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9.2,
            fontweight="bold",
            color="#172026",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.84, "pad": 2.2},
            zorder=10,
        )
    save(fig, "timor_global2_2026_candidate_grids_stacked", dpi=260)

    for grid_km in (10, 5, 1):
        fig, ax = plt.subplots(figsize=(10.5, 4.8))
        error = add_osm_basemap(ax, bounds)
        if error:
            errors.append(error)
        draw_candidate_layer(ax, candidates[grid_km], existing)
        save(fig, f"timor_global2_2026_candidate_grid_{grid_km}km", dpi=300)
    return errors


def make_typical_solution() -> dict:
    grid_km = 1
    threshold_km = 5
    budget = 100
    directory = case_dir(grid_km, threshold_km)
    solutions = read_latest_jsonl(directory / "solutions.jsonl")
    frontier = read_latest_jsonl(directory / "frontier.jsonl")
    solution = solutions[budget]
    point = frontier[budget]

    catalog = pd.read_csv(directory / "candidate_catalog.csv")
    catalog["candidate_id"] = catalog["candidate_id"].astype(str)
    candidate_gdf = gpd.GeoDataFrame(
        catalog,
        geometry=gpd.points_from_xy(catalog["Longitude"], catalog["Latitude"]),
        crs="EPSG:4326",
    ).to_crs(epsg=3857)
    selected_ids = set(map(str, solution["candidate_ids"]))
    selected = candidate_gdf.loc[candidate_gdf["candidate_id"].isin(selected_ids)].copy()
    if len(selected) != int(solution["budget"]):
        raise RuntimeError("Stored solution does not map one-to-one to candidate coordinates")

    sources = pd.read_parquet(resolve_sources_path())
    amenities = sources.loc[sources["source_type"].astype(str) == "amenities"].copy()
    existing = gpd.GeoDataFrame(
        amenities,
        geometry=gpd.points_from_xy(amenities["Longitude"], amenities["Latitude"]),
        crs="EPSG:4326",
    ).to_crs(epsg=3857)
    bounds = padded_bounds([candidate_gdf, existing])

    fig, ax = plt.subplots(figsize=(10.8, 5.0))
    basemap_error = add_osm_basemap(ax, bounds, washout=0.13)
    ax.plot(
        candidate_gdf.geometry.x,
        candidate_gdf.geometry.y,
        linestyle="None",
        marker=".",
        markersize=0.65,
        markeredgewidth=0,
        color=CANDIDATE_COLOR,
        alpha=0.28,
        rasterized=True,
        zorder=3,
    )
    ax.scatter(
        existing.geometry.x,
        existing.geometry.y,
        marker="D",
        s=16,
        facecolor=EXISTING_COLOR,
        edgecolor="white",
        linewidth=0.45,
        alpha=0.92,
        zorder=4,
    )
    ax.scatter(
        selected.geometry.x,
        selected.geometry.y,
        marker="o",
        s=28,
        facecolor=SELECTED_COLOR,
        edgecolor="white",
        linewidth=0.6,
        alpha=0.98,
        zorder=5,
    )
    save(fig, "timor_global2_2026_exact_solution_grid1km_threshold5km_p100", dpi=300)

    metadata = {
        "grid_km": grid_km,
        "threshold_km": threshold_km,
        "budget": budget,
        "selected_count": len(selected),
        "coverage_pct": float(point["coverage_pct"]),
        "total_covered_population": float(point["total_covered_population"]),
        "baseline_coverage_pct": float(
            json.loads((directory / "instance_metadata.json").read_text(encoding="utf-8"))[
                "baseline_coverage_pct"
            ]
        ),
        "basemap_error": basemap_error,
    }
    (FIGURES / "timor_global2_2026_exact_solution_p100_metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
    )
    return metadata


def make_five_km_threshold_overlay() -> None:
    fig, ax = plt.subplots(figsize=(8.6, 4.8))
    maximum_saturation = 0
    series = {}
    for grid_km in (10, 5, 1):
        records = read_latest_jsonl(case_dir(grid_km, 5) / "frontier.jsonl")
        frame = pd.DataFrame(records.values()).sort_values("budget")
        saturation_budget = int(frame["budget"].max())
        maximum_saturation = max(maximum_saturation, saturation_budget)
        series[grid_km] = (frame, saturation_budget)
    x_end = int(math.ceil(maximum_saturation * 1.08 / 50.0) * 50)

    for grid_km in (10, 5, 1):
        frame, saturation_budget = series[grid_km]
        saturation_coverage = float(frame.iloc[-1]["coverage_pct"])
        color = GRID_COLORS[grid_km]
        ax.plot(
            frame["budget"],
            frame["coverage_pct"],
            color=color,
            linewidth=2.0,
            label=f"{grid_km} km candidate grid",
        )
        ax.scatter([saturation_budget], [saturation_coverage], color=color, s=30, zorder=4)
        if saturation_budget < x_end:
            ax.plot(
                [saturation_budget, x_end],
                [saturation_coverage, saturation_coverage],
                color=color,
                linewidth=1.5,
                linestyle="--",
            )
    ax.set_xlim(0, x_end)
    ax.set_xlabel("New facilities")
    ax.set_ylabel("Covered population (%)")
    ax.legend(frameon=False, ncol=3, loc="lower right")
    style_plot(ax)
    save(fig, "timor_global2_2026_exact_pareto_threshold5km_by_grid")


def make_saturation_resolution_figure() -> None:
    summary = pd.read_csv(OUTPUT / "exact_frontier_summary.csv")
    fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.3), constrained_layout=True)
    x = np.arange(3)
    grid_order = [10, 5, 1]
    for threshold_km in (2, 5, 10):
        group = summary.loc[summary["threshold_km"] == threshold_km].set_index("grid_km")
        values = [float(group.loc[grid, "saturation_coverage_pct"]) for grid in grid_order]
        budgets = [int(group.loc[grid, "exact_saturation_budget"]) for grid in grid_order]
        color = THRESHOLD_COLORS[threshold_km]
        axes[0].plot(x, values, marker="o", linewidth=2.0, color=color, label=f"{threshold_km} km threshold")
        axes[1].plot(x, budgets, marker="o", linewidth=2.0, color=color, label=f"{threshold_km} km threshold")
    for ax in axes:
        ax.set_xticks(x, ["10 km", "5 km", "1 km"])
        ax.set_xlabel("Candidate-grid spacing")
        style_plot(ax)
    axes[0].set_ylabel("Exact saturation coverage (%)")
    axes[1].set_ylabel("Exact saturation budget")
    axes[1].set_yscale("log")
    axes[0].legend(frameon=False, loc="best")
    save(fig, "timor_global2_2026_saturation_by_resolution")


def clock_tick(seconds: float, _position: float) -> str:
    seconds = max(float(seconds), 0.0)
    if seconds < 60:
        return f"{seconds:.0f}s"
    minutes, remaining = divmod(int(round(seconds)), 60)
    if minutes < 60:
        return f"{minutes}:{remaining:02d}"
    hours, minutes = divmod(minutes, 60)
    return f"{hours}:{minutes:02d}:{remaining:02d}"


def make_runtime_figure() -> None:
    summary = pd.read_csv(OUTPUT / "exact_frontier_summary.csv")
    summary = summary.sort_values(["grid_km", "threshold_km"], ascending=[True, False])
    labels = [f"{row.grid_km:g} km grid / {row.threshold_km:g} km threshold" for row in summary.itertuples()]
    solve = summary["gurobi_solve_seconds"].to_numpy(float)
    overhead = np.maximum(summary["exact_wall_seconds"].to_numpy(float) - solve, 0.0)
    y = np.arange(len(summary))

    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    ax.barh(y, solve, color="#0072B2", label="Gurobi solve")
    ax.barh(y, overhead, left=solve, color="#B8C2CC", label="checkpoint and session overhead")
    ax.set_yticks(y, labels)
    ax.set_xlabel("Elapsed time")
    ax.xaxis.set_major_formatter(FuncFormatter(clock_tick))
    ax.legend(frameon=False, loc="lower right")
    style_plot(ax)
    save(fig, "timor_global2_2026_exact_runtime_by_case")


def write_figure_manifest(solution_metadata: dict, basemap_errors: list[str]) -> None:
    rows = [
        ("timor_global2_2026_exact_pareto_3x3", "Nine complete exact Pareto frontiers."),
        ("timor_global2_2026_exact_pareto_threshold5km_by_grid", "Exact 5 km-threshold comparison across candidate-grid resolutions."),
        ("timor_global2_2026_saturation_by_resolution", "Coverage ceilings and saturation budgets as spatial resolution increases."),
        ("timor_global2_2026_exact_runtime_by_case", "Exact solve and durable-checkpoint wall times."),
        ("timor_global2_2026_candidate_grids_stacked", "Current candidate grids with OSM health amenities."),
        ("timor_global2_2026_candidate_grid_10km", "Large individual 10 km candidate-grid map."),
        ("timor_global2_2026_candidate_grid_5km", "Large individual 5 km candidate-grid map."),
        ("timor_global2_2026_candidate_grid_1km", "Large individual 1 km candidate-grid map."),
        ("timor_global2_2026_exact_solution_grid1km_threshold5km_p100", "Representative exact p=100 deployment on the 1 km grid."),
    ]
    lines = ["# Timor-Leste WorldPop 2026 figure set", ""]
    for stem, description in rows:
        lines.append(f"- `{stem}.pdf` and `.png`: {description}")
    lines.extend(
        [
            "",
            "## Representative solution",
            "",
            f"The mapped exact solution uses the 1 km candidate grid, a 5 km service threshold, and p={solution_metadata['budget']}. ",
            f"It covers {solution_metadata['coverage_pct']:.3f}% of the population, compared with a baseline of {solution_metadata['baseline_coverage_pct']:.3f}% from existing OSM health amenities.",
            "Blue points are candidate sites, orange diamonds are existing OSM health amenities, and green circles are selected new sites.",
            "",
            f"OSM basemap errors: {len(basemap_errors) + int(solution_metadata['basemap_error'] is not None)}",
        ]
    )
    (FIGURES / "README.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def main() -> None:
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "axes.labelsize": 10,
            "axes.titlesize": 11,
            "legend.fontsize": 9,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
        }
    )
    basemap_errors = make_candidate_maps()
    solution_metadata = make_typical_solution()
    make_five_km_threshold_overlay()
    make_saturation_resolution_figure()
    make_runtime_figure()
    write_figure_manifest(solution_metadata, basemap_errors)
    print(FIGURES)


if __name__ == "__main__":
    main()

In [ ]:
figure_files = sorted((PARETO_OUTPUT / "figures").glob("*.pdf"))
print("Generated PDF figures:", len(figure_files))
for path in figure_files:
    print(" -", path.name)

## Artifact layout

For each of the nine cases:

- `instance_metadata.json`: dimensions, hashes, baseline, ceiling, and preprocessing provenance;
- `candidate_catalog.csv`: stable candidate IDs, coordinates, snapped nodes, and components;
- `frontier.jsonl`: durable detailed timing and objective checkpoint for every budget;
- `frontier.csv`: convenient latest-optimal projection of the checkpoints;
- `solutions.jsonl`: every corresponding exact selected-candidate solution;
- `events.jsonl`: instance-build, greedy-bound, model-build, session, and saturation timing events.

At campaign level, `campaign_manifest.json`, `exact_frontier_summary.csv`, `exact_frontier_validation.csv`, and the PDF/PNG 3-by-3 figure provide the audit and reporting layer.

The `figures` directory contains the complete publication-ready visual set and a `README.md` describing the symbol encoding and representative solution.